In [1]:
import pandas as pd 
import numpy as np

In [2]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [3]:
data = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

In [4]:
df = pd.read_csv(data)
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [5]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [6]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [7]:
for col in df.columns:
    if df[col].dtype in ['string']:
        df[col] = df[col].fillna('NA')
    elif df[col].dtype in ['int64', 'float64']:
        df[col] = df[col].fillna(0)

In [8]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [9]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [10]:
categorical = ['lead_source','industry','employment_status','location']
numerical = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']

Which numerical variable (among the following 4) has the highest AUC?

- lead_score
- number_of_courses_viewed
- interaction_count
- annual_income

In [11]:
for col in numerical:
  score = roc_auc_score(y_train, df_train[col])
  if score < 0.5:
    score = roc_auc_score(y_train, -df_train[col])
  print(f'{col}: {score:.3f}')

annual_income: 0.608
number_of_courses_viewed: 0.723
interaction_count: 0.765
lead_score: 0.788


Apply one-hot-encoding using DictVectorizer and train the logistic regression with these parameters:

In [12]:
X_train_dict = df_train[categorical + numerical].to_dict(orient="records")
X_val_dict = df_val[categorical + numerical].to_dict(orient="records")

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(X_train_dict)
X_val = dv.transform(X_val_dict)

model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
auc_score = roc_auc_score(y_val, y_pred)

print(f"Model AUC: {auc_score:.3f}")


Model AUC: 0.732


Choose the threshold with the smallest absolute difference between precision and recall.

- 0.43
- 0.63
- 0.73
- 0.83

In [13]:
thresholds = np.linspace(0, 1, 101)
scores = []

for t in thresholds:
    predict_positive = (y_pred >= t)
    predict_negative = (y_pred < t)
    
    tp = (predict_positive & (y_val == 1)).sum()
    fp = (predict_positive & (y_val == 0)).sum()
    fn = (predict_negative & (y_val == 1)).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    scores.append((t, precision, recall))

df_scores = pd.DataFrame(scores, columns=['threshold', 'precision', 'recall'])
df_scores['diff'] = abs(df_scores['precision'] - df_scores['recall'])
print(df_scores.sort_values(by='diff').head(10))

     threshold  precision    recall      diff
97        0.97   0.000000  0.000000  0.000000
98        0.98   0.000000  0.000000  0.000000
99        0.99   0.000000  0.000000  0.000000
100       1.00   0.000000  0.000000  0.000000
63        0.63   0.732759  0.725256  0.007503
62        0.62   0.725914  0.745734  0.019820
64        0.64   0.729875  0.696246  0.033629
61        0.61   0.719048  0.773038  0.053990
65        0.65   0.741021  0.668942  0.072079
60        0.60   0.706337  0.779863  0.073527


At which threshold F1 is maximal?

- 0.21
- 0.41
- 0.61
- 0.81

In [14]:
f1_scores = []
for t, p, r in zip(df_scores["threshold"], df_scores["precision"], df_scores["recall"]):
    f1 = ((2 * p * r) / (p + r) if (p + r) > 0 else 0)
    f1_scores.append((t, f1))
df_f1 = pd.DataFrame(f1_scores, columns=["threshold", "f1_score"])
print(df_f1.sort_values(by="f1_score", ascending=False).head(1))

    threshold  f1_score
41       0.41  0.757616


How large is standard deviation of the scores across different folds?

- 0.001
- 0.007
- 0.013
- 0.060

In [15]:
kf = KFold(n_splits=5, shuffle=True, random_state=1)
auc_scores = []

for train_idx, val_idx in kf.split(df_full_train):
    df_train_cv = df_full_train.iloc[train_idx]
    df_val_cv = df_full_train.iloc[val_idx]

    y_train_cv = df_train_cv['converted'].values
    y_val_cv = df_val_cv['converted'].values

    X_train_dict = df_train_cv[categorical + numerical].to_dict(orient='records')
    X_val_dict = df_val_cv[categorical + numerical].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train_cv = dv.fit_transform(X_train_dict)
    X_val_cv = dv.transform(X_val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
    model.fit(X_train_cv, y_train_cv)

    y_pred_cv = model.predict_proba(X_val_cv)[:, 1]
    auc = roc_auc_score(y_val_cv, y_pred_cv)
    auc_scores.append(auc)

print(f"Mean AUC: {np.mean(auc_scores):.3f} , Standard Deviation: {np.std(auc_scores):.3f}")

Mean AUC: 0.732 , Standard Deviation: 0.007


Which C leads to the best mean ROC AUC?

- 0.000001
- 0.001
- 1

In [16]:
C_values = [0.000001, 0.001, 0.1, 1, 10]

for C in C_values:
    auc_scores = []
    for train_idx, val_idx in kf.split(df_full_train):
        df_train_cv = df_full_train.iloc[train_idx]
        df_val_cv = df_full_train.iloc[val_idx]
        
        y_train_cv = df_train_cv['converted'].values
        y_val_cv = df_val_cv['converted'].values
        
        X_train_dict = df_train_cv[categorical + numerical].to_dict(orient='records')
        X_val_dict = df_val_cv[categorical + numerical].to_dict(orient='records')
        
        dv = DictVectorizer(sparse=False)
        X_train_cv = dv.fit_transform(X_train_dict)
        X_val_cv = dv.transform(X_val_dict)
        
        model = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
        model.fit(X_train_cv, y_train_cv)
        
        y_pred_cv = model.predict_proba(X_val_cv)[:, 1]
        auc = roc_auc_score(y_val_cv, y_pred_cv)
        auc_scores.append(auc)
        
    print(f"C: {C} , Mean AUC: {np.mean(auc_scores):.3f} ,Std: {np.std(auc_scores):.3f}")

C: 1e-06 , Mean AUC: 0.617 ,Std: 0.019
C: 0.001 , Mean AUC: 0.749 ,Std: 0.010
C: 0.1 , Mean AUC: 0.732 ,Std: 0.007
C: 1 , Mean AUC: 0.732 ,Std: 0.007
C: 10 , Mean AUC: 0.732 ,Std: 0.007
